# DocMind RAG 系统评测 Notebook

> 🎯 **目的**：用 RAGAS 框架科学评测你的 RAG 系统好不好用

> 📘 **新手必读**：
> - RAGAS = RAG Assessment，专门评测 RAG 系统的开源工具
> - 它会从 4 个维度给你的系统打分（0-1 分，越高越好）
> - 做完评测你就知道哪里需要优化了

---

## RAGAS 评测的 4 个维度（面试必问！）

| 指标 | 英文 | 测什么 | 好的表现 |
|------|------|--------|----------|
| 忠实度 | Faithfulness | 回答是否基于文档？有没有编造？ | > 0.80 |
| 答案相关性 | Answer Relevancy | 有没有答非所问？废话多不多？ | > 0.80 |
| 上下文相关性 | Context Relevancy | 检索到的文档和问题相关吗？ | > 0.70 |
| 上下文召回率 | Context Recall | 有没有遗漏关键信息？ | > 0.70 |

> 💡 **面试技巧**：面试官问你"怎么评价你的RAG系统"时，
> 直接说出这 4 个维度和它们的含义，比说"用户反馈不错"专业 10 倍。

In [ ]:
# ============================================================
# 第一步：安装依赖（首次运行取消注释下面这行）
# ============================================================
# !pip install ragas>=0.2.0 pandas datasets

import sys
from pathlib import Path

# 把项目根目录加入 Python 路径
# 为什么需要这行：Jupyter 默认只在当前目录找模块，
# 加上后能从 notebook 里导入 src/ 下的代码
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"📂 项目根目录: {PROJECT_ROOT}")

In [ ]:
# ============================================================
# 第二步：加载 DocMind 的各个组件
# ============================================================

from src.config import config
from src.embeddings import Embedder
from src.retrieval import VectorStore, BM25Retriever, HybridRetriever
from src.generation import LLMClient, build_rag_prompt, format_context

print("📥 加载嵌入模型...")
embedder = Embedder(device=config.EMBED_DEVICE)

print("📦 连接向量数据库...")
vector_store = VectorStore(
    persist_dir=config.CHROMA_PERSIST_DIR,
    collection_name=config.CHROMA_COLLECTION_NAME,
)

print("📇 加载 BM25 检索器...")
bm25 = BM25Retriever()
import os
bm25_path = os.path.join(config.CHROMA_PERSIST_DIR, "bm25_index.pkl")
if os.path.exists(bm25_path):
    bm25.load(bm25_path)
    print(f"   ✅ BM25 索引加载成功")
else:
    print(f"   ⚠️ BM25 索引不存在，请先运行: python scripts/ingest_docs.py --dir data/sample_docs/")

print("🔀 初始化混合检索器...")
hybrid = HybridRetriever(vector_store, bm25)

print("🤖 初始化 LLM 客户端...")
llm = LLMClient(
    api_key=config.DEEPSEEK_API_KEY,
    base_url=config.DEEPSEEK_BASE_URL,
    model=config.DEEPSEEK_MODEL,
)

print(f"\n✅ 所有组件加载完成！")
print(f"   向量库文档数: {vector_store.count()}")
if vector_store.count() == 0:
    print(f"   ⚠️ 知识库为空！请先导入文档:")
    print(f"      python scripts/ingest_docs.py --dir data/sample_docs/")

In [ ]:
# ============================================================
# 第三步：定义一个完整的 RAG 问答函数
# ============================================================
# RAGAS 需要的是：输入 question → 输出 answer + contexts
# 我们把 DocMind 的 RAG 管道包装成 RAGAS 需要的格式

def docmind_rag(question: str) -> dict:
    """
    DocMind RAG 管道——给它一个问题，它返回答案 + 引用来源
    """
    # 1. 生成查询向量
    query_embedding = embedder.embed(question)
    
    # 2. 混合检索（BM25 + 向量）
    search_results = hybrid.search(question, query_embedding, top_k=5)
    
    # 3. 提取检索到的文档文本（这些就是 RAGAS 评测用的 contexts）
    contexts = [r["text"] for r in search_results]
    
    if not search_results:
        return {
            "answer": "知识库中未找到相关信息。",
            "contexts": [],
        }
    
    # 4. 构建 Prompt
    context = format_context(search_results)
    messages = build_rag_prompt(question=question, context=context)
    
    # 5. 调用 LLM 生成答案
    answer = llm.chat(messages, temperature=0.1)
    #    ↑ temperature=0.1：让模型输出更确定，评测更稳定
    
    return {
        "answer": answer,
        "contexts": contexts,
    }

print("✅ RAG 函数定义完成")
print("   可以测试: docmind_rag('什么是SKU？')")

In [ ]:
# ============================================================
# 第四步：快速测试——看看系统能不能正常回答
# ============================================================

if vector_store.count() > 0:
    test_question = "SKU12345的库存还够卖几天？"
    print(f"❓ 测试问题: {test_question}")
    print("-" * 40)
    
    result = docmind_rag(test_question)
    print(f"📝 回答:\n{result['answer'][:500]}")
    print(f"\n📚 检索到 {len(result['contexts'])} 个相关文档片段")
else:
    print("⚠️ 知识库为空，跳过测试")
    print("   请先运行: python scripts/ingest_docs.py --dir data/sample_docs/")

In [ ]:
# ============================================================
# 第五步：准备评测数据集
# ============================================================
# RAGAS 需要一个评测集，包含：
# - question: 用户问题
# - ground_truth: 人工标注的标准答案（参考答案）
# - answer: 系统实际生成的答案
# - contexts: 检索到的文档片段

# 📝 下面是我们准备的 10 个评测问题
# 每个问题都有人工写的标准答案（ground_truth），用来衡量系统表现

eval_questions = [
    {
        "question": "什么是SKU？",
        "ground_truth": "SKU是库存量单位（Stock Keeping Unit）的缩写，是供应链管理中最基础的库存标识单位，每个SKU对应一个唯一的物料编号，用于追踪库存、销售和采购。"
    },
    {
        "question": "安全库存的计算公式是什么？",
        "ground_truth": "安全库存 = Z × σ_demand × √LT，其中Z是服务水平系数，σ_demand是需求标准差，LT是提前期。A类物料Z值用1.96(97.5%)。"
    },
    {
        "question": "SKU12345的当前库存情况如何？",
        "ground_truth": "SKU12345当前库存200件，安全库存50件，可售库存150件，日均销量30件，可支撑5天。供应商交期7个工作日，存在约2天供应缺口。"
    },
    {
        "question": "经济订货量（EOQ）的公式是什么？",
        "ground_truth": "EOQ = √(2DS/H)，其中D是年需求量，S是每次订货成本，H是单位年持有成本。"
    },
    {
        "question": "什么是先进先出（FIFO）？",
        "ground_truth": "FIFO是First In First Out的缩写，是库存出库原则：先入库的物料先出库，适用于有保质期的商品如食品、药品、电子产品。"
    },
    {
        "question": "华东仓库6月的库存周转天数是多少？",
        "ground_truth": "华东仓库6月库存周转天数为26.2天，比5月的28.5天减少了2.3天。"
    },
    {
        "question": "SKU12345的替代品是什么？价格差异多少？",
        "ground_truth": "SKU12345的替代品是SKU67890，替代比例为1:1，价格贵10%，功能相似但规格略高。"
    },
    {
        "question": "补货周期包含哪些环节？",
        "ground_truth": "补货周期包括：订单处理时间(1-2个工作日)、供应商生产时间、运输时间(国内2-5天，国际7-30天)、质检入库时间(1-2个工作日)。"
    },
    {
        "question": "库存周转率的目标是多少？",
        "ground_truth": "库存周转率(Inventory Turnover)的目标值是大于12次/年，公式为出库金额除以平均库存金额。"
    },
    {
        "question": "WMS是什么系统？公司用的是什么？",
        "ground_truth": "WMS是仓库管理系统(Warehouse Management System)，用于管理仓库日常运作包括入出库和盘点。公司当前使用通天晓WMS V4.2。"
    },
]

print(f"📊 准备了 {len(eval_questions)} 个评测问题")
print(f"   其中包含: 概念定义类、数值计算类、库存查询类、多步推理类")

In [ ]:
# ============================================================
# 第六步：跑评测！用 RAGAS 给系统打分
# ============================================================

from datasets import Dataset
import pandas as pd

if vector_store.count() == 0:
    print("❌ 知识库为空，无法评测。请先导入文档。")
else:
    # 对每个问题跑 RAG，收集结果
    print("🔄 正在对每个问题执行 RAG 检索 + 生成...")
    
    eval_data = {
        "question": [],
        "ground_truth": [],
        "answer": [],
        "contexts": [],
    }
    
    for i, item in enumerate(eval_questions):
        print(f"   [{i+1}/{len(eval_questions)}] {item['question'][:40]}...")
        result = docmind_rag(item["question"])
        eval_data["question"].append(item["question"])
        eval_data["ground_truth"].append(item["ground_truth"])
        eval_data["answer"].append(result["answer"])
        eval_data["contexts"].append(result["contexts"])
    
    print(f"✅ 全部 {len(eval_questions)} 个问题已完成！")
    
    # 转为 RAGAS 需要的 Dataset 格式
    eval_dataset = Dataset.from_dict(eval_data)
    print(f"\n📊 评测数据集已构建")
    print(f"   样例答案预览:\n   {eval_data['answer'][0][:200]}...")

In [ ]:
# ============================================================
# 第七步：计算 RAGAS 指标
# ============================================================
# ⚠️ RAGAS 评测需要调用 LLM 来判断质量（用 LLM 评测 LLM）
#    所以它会消耗一些 API 调用费用。

if vector_store.count() > 0:
    from ragas import evaluate
    from ragas.metrics import (
        faithfulness,         # 忠实度：回答是否来自文档
        answer_relevancy,     # 答案相关性：有没有跑题
        context_recall,       # 上下文召回率：关键信息有没有搜到
        context_precision,    # 上下文精确率：搜到的文档有没有用
    )
    
    # 配置 RAGAS 使用 DeepSeek 作为评判模型
    # RAGAS 默认用 OpenAI，这里改为用 DeepSeek
    from ragas.llms import LangchainLLMWrapper
    from langchain_openai import ChatOpenAI
    from ragas.embeddings import LangchainEmbeddingsWrapper
    from langchain_openai import OpenAIEmbeddings
    
    # ⚠️ 这里需要你的 DeepSeek API Key
    # 如果没有设置，请先设置环境变量 DEEPSEEK_API_KEY
    # 或者在 config.py 中填写
    
    try:
        evaluator_llm = LangchainLLMWrapper(ChatOpenAI(
            model=config.DEEPSEEK_MODEL,
            openai_api_key=config.DEEPSEEK_API_KEY,
            openai_api_base=config.DEEPSEEK_BASE_URL,
            temperature=0,
        ))
        
        print("📊 正在计算 RAGAS 评测指标...")
        print("   （这可能需要 1-2 分钟，因为每个问题都需要 LLM 评判）")
        
        result = evaluate(
            eval_dataset,
            metrics=[
                faithfulness,
                answer_relevancy,
                context_recall,
                context_precision,
            ],
            llm=evaluator_llm,
        )
        
        # 转换为 DataFrame 方便查看
        result_df = result.to_pandas()
        print("\n" + "="*60)
        print("📊 RAGAS 评测结果")
        print("="*60)
        
        # 计算平均分
        metrics_display = {
            "忠实度 (Faithfulness)": result_df["faithfulness"].mean() if "faithfulness" in result_df.columns else 0,
            "答案相关性 (Answer Relevancy)": result_df["answer_relevancy"].mean() if "answer_relevancy" in result_df.columns else 0,
            "上下文召回率 (Context Recall)": result_df["context_recall"].mean() if "context_recall" in result_df.columns else 0,
            "上下文精确率 (Context Precision)": result_df["context_precision"].mean() if "context_precision" in result_df.columns else 0,
        }
        
        for name, score in metrics_display.items():
            bar = "█" * int(score * 20) + "░" * (20 - int(score * 20))
            print(f"   {name}: {bar} {score:.3f}")
        
        # 整体评分
        overall = sum(metrics_display.values()) / len(metrics_display)
        print(f"\n   🌟 综合评分: {overall:.3f}")
        
        if overall >= 0.80:
            print("   ✅ 系统表现优秀！")
        elif overall >= 0.65:
            print("   ⚠️ 系统表现不错，还有优化空间")
        else:
            print("   ❌ 系统需要优化，请检查检索和生成质量")
        
        print("\n📝 各问题详情:")
        display(result_df[['question', 'faithfulness', 'answer_relevancy']])
        
        result_df.to_csv("eval_results.csv", index=False)
        print(f"\n💾 结果已保存到 eval_results.csv")
        
    except Exception as e:
        print(f"❌ RAGAS 评测出错: {e}")
        print(f"\n💡 可能的原因:")
        print(f"   1. DEEPSEEK_API_KEY 未设置")
        print(f"   2. RAGAS 版本不兼容（需要 >= 0.2.0）")
        print(f"   3. 网络问题无法访问 DeepSeek API")

In [ ]:
# ============================================================
# 附加实验：对比不同分块策略的效果
# ============================================================
# 面试时可以说"我做了分块策略的对比实验"——非常加分！

from src.document_processing import get_chunker

def compare_chunk_strategies(query: str = "SKU12345库存"):
    """
    对比三种分块策略在同一个查询下的检索结果
    """
    # 读取样本文档
    doc_path = PROJECT_ROOT / "data" / "sample_docs" / "库存月报_202606.md"
    if not doc_path.exists():
        print(f"❌ 找不到样本文档: {doc_path}")
        return
    
    content = doc_path.read_text()
    
    for strategy in ["fixed", "recursive"]:
        chunker = get_chunker(strategy)
        chunks = chunker.chunk(content, chunk_size=256, overlap=32)
        print(f"\n📌 策略: {strategy}")
        print(f"   分块数: {len(chunks)}")
        print(f"   平均块大小: {sum(len(c.text) for c in chunks) // max(len(chunks), 1)} 字符")
        
        # 看前 3 个块的摘要
        for i, chunk in enumerate(chunks[:3]):
            print(f"   [块{i}]: {chunk.text[:80]}...")

compare_chunk_strategies()

## 📝 评测总结

跑完这个 notebook 后，你应该有：

1. **RAGAS 四维评分**：知道你的系统在忠实度、相关性、召回率、精确率上的表现
2. **各问题详情**：哪些问题答得好，哪些答得不好
3. **分块策略对比**：固定分块 vs 递归分块的差异

### 🎯 如何解读结果 & 优化建议

| 现象 | 可能原因 | 优化方向 |
|------|----------|----------|
| 忠实度低 | LLM 在编造内容 | 调整 temperature、改进 Prompt 约束 |
| 答案相关性低 | 检索结果噪音多 | 减小 top_k、提高检索精度 |
| 上下文召回率低 | 关键文档没搜到 | 调整分块大小、试试语义分块 |
| 上下文精确率低 | 搜到太多无关文档 | 加元数据过滤、reranking |

### 📖 面试话术

"我在 DocMind 项目中用 RAGAS 框架对系统做了系统性评测，
从忠实度、答案相关性、上下文召回率和精确率四个维度打分。
最终综合得分 X.X，其中[最弱的指标]得分偏低，
我通过[具体优化]将其从 X 提升到了 Y。"